In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt
import scipy.io
import torcwa
from tqdm.notebook import tqdm
from pvlib import spectrum
from refractiveindex import RefractiveIndexMaterial
%load_ext line_profiler

# Hardware
# If GPU support TF32 tensor core, the matmul operation is faster than FP32 but with less precision.
# If you need accurate operation, you have to disable the flag below.
torch.backends.cuda.matmul.allow_tf32 = False
sim_dtype = torch.complex64
geo_dtype = torch.float32
device = torch.device('cuda')

def get_sine_eps(x,params,grating_period,eps):
    """Generate sine grating permittivity profile.

    Args:
        x (torch.tensor): 1D tensor of x positions.
        params (torch.Tensor): list of amplitude and phase shift. shape (n,2), where n is n*2*np.pi/grating_period'th frequency.
        eps (float): Permittivity of high-index material.

    Returns:
        torch.tensor: 1D tensor of permittivity profile.
    """
    A = torch.sum(params[:,0]) + 1e-9
    cosines = torch.cos(2.*np.pi*torch.arange(1, params.shape[0]+1, 
                                              dtype=geo_dtype,device=device).unsqueeze(1)*(x.unsqueeze(0)/grating_period)
                                               - params[:,1].unsqueeze(1))
    cosines = cosines * params[:,0].unsqueeze(1)
    eps = 1 + (eps-1)*(0.5*(A+torch.sum(cosines, dim=0))/A)
    return eps.unsqueeze(1)   # make shape (nx,1) so add_layer accepts it

# Simulation environment
# light
spectra = spectrum.get_reference_spectra()
am15g = spectra['global']
wavelengths = torch.arange(350,1110,10,dtype=geo_dtype,device=device)  # nm
sun_weights = torch.tensor(am15g[wavelengths.cpu().numpy()].to_numpy())

# material
si = RefractiveIndexMaterial('main', book='Si', page='Green-2008')
si_eps = torch.tensor(si.get_refractive_index(wavelengths.cpu().numpy()) +
                      1j * si.get_extinction_coefficient(wavelengths.cpu().numpy()))**2

# geometry
h = 1000 #nm
grating_period = 1000 # nm
L = [grating_period, 1.]  # nm
torcwa.rcwa_geo.dtype = geo_dtype
torcwa.rcwa_geo.device = device
torcwa.rcwa_geo.Lx = L[0]
torcwa.rcwa_geo.Ly = L[1]
#torcwa.rcwa_geo.nx = 1000 #done in function call
torcwa.rcwa_geo.ny = 1
#torcwa.rcwa_geo.grid() #done in function call

Order_N = 40
order = [Order_N,0]

#z = torch.linspace(-h,1.5*h,501,device=device)

#x_axis = torcwa.rcwa_geo.x.cpu()
#y_axis = torcwa.rcwa_geo.y.cpu()
#z_axis = z.cpu()

In [ ]:
def get_absorptance(params,wavelength,inc_ang,azi_ang):
    #setup simulation
    torcwa.rcwa_geo.nx = 5000 #set sine grating grid
    torcwa.rcwa_geo.grid()
    A = torch.sum(params[:,0])
    sine_eps = get_sine_eps(torcwa.rcwa_geo.x,params=params,grating_period=grating_period,eps=si_eps[2])
    sim = torcwa.rcwa(freq=1/wavelength,order=order,L=L,dtype=sim_dtype,device=device)
    sim.add_input_layer()
    sim.add_output_layer()
    sim.set_incident_angle(inc_ang=inc_ang,azi_ang=azi_ang)
    sim.add_layer(thickness=A,eps=sine_eps)
    sim.add_layer(thickness=h,eps=si_eps[2])
    sim.solve_global_smatrix()

    # choose probe planes (just above / below film). use same device dtype as sim
    z_top = torch.clone(A)  # e.g. top of film
    z_bot = torch.clone(A+h)  # e.g. bottom of film
    z_air = torch.tensor(0,device=sim._device, dtype=geo_dtype)
    torcwa.rcwa_geo.nx = 5000 #set sampling grid for fields
    torcwa.rcwa_geo.grid()
    P_absorbed_film = torch.zeros(2,device=sim._device, dtype=geo_dtype) # to store absorbed power in film, first item is x polarization, second y polarization
    P_absorbed_grating = torch.zeros(2,device=sim._device, dtype=geo_dtype) # to store absorbed power in grating, first item is x polarization, second y polarization
    A_film = torch.zeros(2,device=sim._device, dtype=geo_dtype) #to store absorptance in film. first item is x polarization, second y polarization
    A_grating = torch.zeros(2,device=sim._device, dtype=geo_dtype) #to store absorptance in grating. first item is x polarization, second y polarization
    P_slices = torch.zeros((2,3),device=sim._device, dtype=geo_dtype) # to store Poynting vector slices at each plane for both polarizations

    sim.source_planewave(amplitude=[1.,0.],direction='forward',notation='xy')
    # request fields at both planes: x_axis is your x sampling (1D tensor), y0 is y coordinate (often 0)
    [Ex, Ey, Ez], [Hx, Hy, Hz] = sim.field_xz(torcwa.rcwa_geo.x, torch.stack((z_top,z_bot,z_air)), y=0.0)
    # Ex,Hy shapes: (nx, 2)  (nx across x, 2 planes)
    S_z = 0.5 * torch.real(Ex * torch.conj(Hy) - Ey * torch.conj(Hx))   # shape (nx,3)
    P_top = torch.trapezoid(S_z[:,0], torcwa.rcwa_geo.x)
    P_bot = torch.trapezoid(S_z[:,1], torcwa.rcwa_geo.x)
    P_air = torch.trapezoid(S_z[:,2], torcwa.rcwa_geo.x)
    P_absorbed_film[0] = P_top - P_bot
    P_absorbed_grating[0] = P_air - P_top
    A_film[0] = P_absorbed_film[0] / P_air
    A_grating[0] = P_absorbed_grating[0] / P_air
    P_slices[0,:] = torch.tensor([P_top, P_bot, P_air], device=sim._device, dtype=geo_dtype)

    sim.source_planewave(amplitude=[0.,1.],direction='forward',notation='xy')
    # request fields at both planes: x_axis is your x sampling (1D tensor), y0 is y coordinate (often 0)
    [Ex, Ey, Ez], [Hx, Hy, Hz] = sim.field_xz(torcwa.rcwa_geo.x, torch.stack((z_top,z_bot,z_air)), y=0.0)
    # Ex,Hy shapes: (nx, 2)  (nx across x, 2 planes)
    S_z = 0.5 * torch.real(Ex * torch.conj(Hy) - Ey * torch.conj(Hx))   # shape (nx,3)
    P_top = torch.trapezoid(S_z[:,0], torcwa.rcwa_geo.x)
    P_bot = torch.trapezoid(S_z[:,1], torcwa.rcwa_geo.x)
    P_air = torch.trapezoid(S_z[:,2], torcwa.rcwa_geo.x)
    P_absorbed_film[1] = P_top - P_bot
    P_absorbed_grating[1] = P_air - P_top
    A_film[1] = P_absorbed_film[1] / P_air
    A_grating[1] = P_absorbed_grating[1] / P_air
    P_slices[1,:] = torch.tensor([P_top, P_bot, P_air], device=sim._device, dtype=geo_dtype)

    return P_absorbed_film, P_absorbed_grating, A_film, A_grating, P_slices

In [ ]:
params = torch.tensor([[20.,0.],[20.,np.pi],[20.,0.]],dtype=geo_dtype,device=device)  # amplitude (nm), phase (rad)
wavelength = wavelengths[2]   # nm
inc_ang = 0.*(np.pi/180)    # radian
azi_ang = 0.*(np.pi/180)    # radian
P_film, P_grating,A_film,A_grating,P_slices = get_absorptance(params,wavelength,inc_ang,azi_ang)
print(f'Absorbed Power in Film: {P_film[0]:.6f}')
print(f'Absorbed Power in Grating: {P_grating[0]:.6f}')
print(f'Absorptance in Film: {A_film[0]:.6f}')
print(f'Absorptance in Grating: {A_grating[0]:.6f}')

In [ ]:
%%timeit -n 10 -r 10
get_absorptance(params,wavelength,inc_ang,azi_ang)